# Notebook 2 — Análisis Exploratorio de Datos (EDA) 📊
**TFG: Detección de Intrusiones a Gran Escala utilizando Machine Learning Distribuido y Big Data**

**Autor:** Eduardo Morillas Rodríguez

**Objetivo:** Exploración visual exhaustiva del dataset CSE-CIC-IDS 2018.
~30 gráficos organizados en 7 bloques temáticos.

**Entrada:** Parquet limpio de `DATA_PARQUET_PATH` (salida del NB1)

**Salida:** Gráficos en `FIGURES_PATH` (cuerpo) y `ANNEX_PATH` (anexos)

---
## Imports y Configuración

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import re
import matplotlib.ticker as ticker
from scipy import stats
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

# missingno: librería estándar para auditoría visual de datos faltantes
try:
	import missingno as msno
	HAS_MISSINGNO = True
except ImportError:
	HAS_MISSINGNO = False
	print("⚠️ missingno no disponible. Se usará heatmap estándar como fallback.")

from config import *

spark = get_spark_session("TFG_IDS_NB2_EDA")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 21:32:01 WARN Utils: Your hostname, eespcachcpro01, resolves to a loopback address: 127.0.1.1; using 10.151.52.2 instead (on interface ens3f0)
26/05/03 21:32:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 21:32:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 21:32:01 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
# === CARGAR DATOS DESDE PARQUET ===
df = spark.read.parquet(DATA_PARQUET_PATH)
total_rows = df.count()
print(f"📊 Dataset cargado: {total_rows:,} filas × {len(df.columns)} columnas")

numeric_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, (DoubleType, FloatType, IntegerType, LongType))
]
print(f"   Columnas numéricas: {len(numeric_cols)}")

print("\n🔍 Calculando varianza para seleccionar top features...")
variances = {}
for col_name in numeric_cols:
    var_val = df.select(F.variance(col_name)).first()[0]
    if var_val is not None and var_val > 0:
        variances[col_name] = var_val

TOP_N = 10
top_features = sorted(variances, key=variances.get, reverse=True)[:TOP_N]
print(f"   Top {TOP_N} features por varianza: {top_features}")

📊 Dataset cargado: 16,232,943 filas × 84 columnas
   Columnas numéricas: 79

🔍 Calculando varianza para seleccionar top features...


[Stage 238:=====================================>               (88 + 36) / 124]

   Top 10 features por varianza: ['Fwd IAT Min', 'Flow IAT Min', 'Flow IAT Max', 'Fwd IAT Max', 'Idle Max', 'Flow Duration', 'Fwd IAT Tot', 'Fwd IAT Std', 'Flow IAT Std', 'Idle Mean']


---
## 2.1 — Distribución de Clases (Target)

In [3]:
# === GRÁFICO 1: Bar chart — distribución absoluta ===
print("📊 Gráfico 1: Distribución de clases (valores absolutos)")

label_counts = (
    df.groupBy("Label").count()
    .orderBy(F.desc("count"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(16, 8))
colors = [COLOR_PALETTE.get(label, "#95a5a6") for label in label_counts["Label"]]
bars = ax.barh(label_counts["Label"][::-1], label_counts["count"][::-1], color=colors[::-1])
ax.set_xlabel("Número de registros")
ax.set_title("Distribución de Clases — CSE-CIC-IDS 2018")
for bar in bars:
    width = bar.get_width()
    ax.text(width + total_rows * 0.005, bar.get_y() + bar.get_height() / 2,
            f"{int(width):,}", ha="left", va="center", fontsize=10)
plt.tight_layout()
save_figure(fig, "01_distribucion_clases_absoluta")

📊 Gráfico 1: Distribución de clases (valores absolutos)


  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/01_distribucion_clases_absoluta.png


In [4]:
# === GRÁFICO 2: Bar chart — escala logarítmica ===
print("📊 Gráfico 2: Distribución de clases (escala logarítmica)")

fig, ax = plt.subplots(figsize=(16, 8))
ax.barh(label_counts["Label"][::-1], label_counts["count"][::-1], color=colors[::-1])
ax.set_xscale("log")
ax.set_xlabel("Número de registros (escala log)")
ax.set_title("Distribución de Clases — Escala Logarítmica")
plt.tight_layout()
save_figure(fig, "02_distribucion_clases_log")

📊 Gráfico 2: Distribución de clases (escala logarítmica)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/02_distribucion_clases_log.png


In [5]:
# === GRÁFICO 3: Donut chart ===
# Las clases con <2% de representación se agrupan en "Otros ataques"
# para evitar solapamiento de etiquetas en un dataset con distribución long-tail.
print("📊 Gráfico 3: Donut chart (proporción relativa)")

total_count = label_counts["count"].sum()
label_counts["pct"] = label_counts["count"] / total_count * 100

main_classes_df = label_counts[label_counts["pct"] >= 2.0].copy()
minor_classes_df = label_counts[label_counts["pct"] < 2.0].copy()

if len(minor_classes_df) > 0:
    otros_row = pd.DataFrame({
        "Label": ["Otros ataques"],
        "count": [minor_classes_df["count"].sum()],
        "pct": [minor_classes_df["pct"].sum()]
    })
    donut_data = pd.concat([main_classes_df[["Label", "count", "pct"]], otros_row], ignore_index=True)
else:
    donut_data = main_classes_df[["Label", "count", "pct"]].copy()

donut_colors = [
    COLOR_PALETTE.get(l, "#95a5a6") if l != "Otros ataques" else "#bdc3c7"
    for l in donut_data["Label"]
]
explode = [0.02] * len(donut_data)

fig, ax = plt.subplots(figsize=(12, 12))
wedges, texts, autotexts = ax.pie(
    donut_data["count"],
    labels=donut_data["Label"],
    autopct=lambda pct: f"{pct:.1f}%" if pct > 0.5 else "",
    colors=donut_colors,
    explode=explode,
    pctdistance=0.82, startangle=90,
    textprops={"fontsize": 11},
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight("bold")

centre_circle = plt.Circle((0, 0), 0.60, fc="white")
ax.add_artist(centre_circle)
ax.set_title("Proporción de Cada Clase de Tráfico\n(clases < 2% agrupadas en 'Otros ataques')",
            fontsize=14)
plt.tight_layout()
save_figure(fig, "03_donut_clases")

# Tabla auxiliar con desglose de "Otros ataques"
if len(minor_classes_df) > 0:
    print("\n📋 Desglose de 'Otros ataques' (clases con <2% del total):")
    print("-" * 55)
    for _, row in minor_classes_df.iterrows():
        print(f"   {row['Label']:<30s} {int(row['count']):>10,}  ({row['pct']:.2f}%)")
    print("-" * 55)
    print(f"   {'TOTAL Otros':<30s} {int(minor_classes_df['count'].sum()):>10,}  "
        f"({minor_classes_df['pct'].sum():.2f}%)")

📊 Gráfico 3: Donut chart (proporción relativa)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/03_donut_clases.png

📋 Desglose de 'Otros ataques' (clases con <2% del total):
-------------------------------------------------------
   Bot                               286,191  (1.76%)
   FTP-BruteForce                    193,360  (1.19%)
   SSH-Bruteforce                    187,589  (1.16%)
   Infilteration                     161,934  (1.00%)
   DoS attacks-SlowHTTPTest          139,890  (0.86%)
   DoS attacks-GoldenEye              41,508  (0.26%)
   DoS attacks-Slowloris              10,990  (0.07%)
   DDoS attack-LOIC-UDP                1,730  (0.01%)
   Brute Force -Web                      611  (0.00%)
   Brute Force -XSS                      230  (0.00%)
   SQL Injection                          87  (0.00%)
-------------------------------------------------------
   TOTAL Otros                     1,024,120  (6.31%)


In [6]:
# === GRÁFICO 4: Treemap interactivo (Plotly) ===
print("📊 Gráfico 4: Treemap interactivo (Plotly)")

fig_treemap = px.treemap(
    label_counts, path=["Label"], values="count",
    color="count", color_continuous_scale="RdYlGn_r",
    title="Treemap — Volumen por Tipo de Ataque",
)
fig_treemap.update_layout(width=1000, height=700)
fig_treemap.write_html(os.path.join(FIGURES_PATH, "04_treemap_clases.html"))
try:
    fig_treemap.write_image(os.path.join(FIGURES_PATH, "04_treemap_clases.png"), scale=2)
    print("  💾 Treemap guardado (HTML + PNG)")
except Exception:
    print("  💾 Treemap guardado (solo HTML — incluir screenshot en el PDF)")

📊 Gráfico 4: Treemap interactivo (Plotly)
  💾 Treemap guardado (solo HTML — incluir screenshot en el PDF)


---
## 2.2 — Distribuciones Univariantes de Features

In [7]:
# === MUESTRA PARA GRÁFICOS ===
SAMPLE_SIZE = min(100_000, total_rows)
sample_fraction = SAMPLE_SIZE / total_rows
df_sample_pd = df.select(top_features + ["Label"]).sample(False, sample_fraction, seed=SEED).toPandas()

all_labels = df_sample_pd["Label"].unique().tolist()

In [8]:
# === GRÁFICO 5a: Histogramas + KDE — Escala completa ===
# Se generan dos versiones: escala completa (para evidenciar outliers)
# y winsorizada (para revelar la forma real de la distribución central).
print("📊 Gráfico 5a: Histogramas + KDE (escala completa)")

n_features = len(top_features)
n_cols_grid = 3
n_rows_grid = (n_features + n_cols_grid - 1) // n_cols_grid

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(18, 4 * n_rows_grid))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    data = df_sample_pd[feat].dropna()
    ax.hist(data, bins=50, density=True, alpha=0.7, color="#3498db", edgecolor="white")
    try:
        data.plot.kde(ax=ax, color="#e74c3c", linewidth=2)
    except Exception:
        pass
    ax.set_title(f"{feat}\n[{data.min():.2e}, {data.max():.2e}]", fontsize=10)
    ax.set_ylabel("Densidad")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuciones Univariantes — Escala Completa", fontsize=16, y=1.02)
plt.tight_layout()
save_figure(fig, "05a_histogramas_kde_completo")

📊 Gráfico 5a: Histogramas + KDE (escala completa)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/05a_histogramas_kde_completo.png


In [9]:
# === GRÁFICO 5b: Histogramas + KDE — Winsorizado P0.5–P99.5 ===
print("📊 Gráfico 5b: Histogramas + KDE (winsorizado P0.5–P99.5)")

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(18, 4 * n_rows_grid))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    data = df_sample_pd[feat].dropna()
    q_low, q_high = data.quantile(0.005), data.quantile(0.995)
    data_win = data[(data >= q_low) & (data <= q_high)]
    pct_clip = (1 - len(data_win) / len(data)) * 100
    ax.hist(data_win, bins=50, density=True, alpha=0.7, color="#2ecc71", edgecolor="white")
    try:
        data_win.plot.kde(ax=ax, color="#e74c3c", linewidth=2)
    except Exception:
        pass
    ax.set_title(f"{feat} — P0.5–P99.5\n({pct_clip:.1f}% datos excluidos)", fontsize=10)
    ax.set_ylabel("Densidad")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuciones Univariantes — Winsorizado P0.5–P99.5", fontsize=16, y=1.02)
plt.tight_layout()
save_figure(fig, "05b_histogramas_kde_winsorizado")

📊 Gráfico 5b: Histogramas + KDE (winsorizado P0.5–P99.5)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/05b_histogramas_kde_winsorizado.png


In [22]:
# === GRÁFICO 6: Violin plots ===
# Se separan las clases en dos paneles según su varianza, con escala Y
# independiente, para ver la estructura interna de cada grupo sin que
# las clases dominantes aplaste a las minoritarias.
print("📊 Gráfico 6: Violin plots (facetado por grupo de varianza)")

violin_features = top_features[:4]
high_var_classes = [c for c in ["Benign", "DDoS attacks-LOIC-HTTP", "Infilteration"] if c in all_labels]
low_var_classes = [c for c in ["DoS attacks-Hulk", "DDoS attack-HOIC", "SSH-Bruteforce",
                                "FTP-BruteForce", "Bot"] if c in all_labels]

for feat_idx, feat in enumerate(violin_features):
    fig, (ax_high, ax_low) = plt.subplots(1, 2, figsize=(20, 7), sharey=False)

    data_h = df_sample_pd[[feat, "Label"]].dropna()
    data_h = data_h[data_h["Label"].isin(high_var_classes)]
    if len(data_h) > 0:
        sns.violinplot(data=data_h, x="Label", y=feat, hue="Label",
                    palette=COLOR_PALETTE, inner="quartile", cut=0,
                    legend=False, ax=ax_high)
    ax_high.set_title(f"{feat} — Clases alta varianza", fontsize=12)
    ax_high.tick_params(axis="x", rotation=30)
    ax_high.set_xlabel("")

    data_l = df_sample_pd[[feat, "Label"]].dropna()
    data_l = data_l[data_l["Label"].isin(low_var_classes)]
    if len(data_l) > 0:
        sns.violinplot(data=data_l, x="Label", y=feat, hue="Label",
                    palette=COLOR_PALETTE, inner="quartile", cut=0,
                    legend=False, ax=ax_low)
    ax_low.set_title(f"{feat} — Clases baja varianza (escala Y independiente)", fontsize=12)
    ax_low.tick_params(axis="x", rotation=30)
    ax_low.set_xlabel("")

    fig.suptitle(f"Violin Plots — {feat}", fontsize=15, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"06_violin_{feat_idx+1}_{feat.replace(' ', '_')}")



📊 Gráfico 6: Violin plots (facetado por grupo de varianza)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/06_violin_1_Fwd_IAT_Min.png
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/06_violin_2_Flow_IAT_Min.png
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/06_violin_3_Flow_IAT_Max.png
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/06_violin_4_Fwd_IAT_Max.png


In [23]:
# === GRÁFICO 7: Box plots ===
# Doble perspectiva: fila superior con datos completos (outliers extremos visibles)
# y fila inferior con recorte P1–P99 (estructura central de la distribución).
print("📊 Gráfico 7: Box plots (doble perspectiva)")

box_features = top_features[:4]
top_classes_box = label_counts["Label"].head(8).tolist()

fig, axes = plt.subplots(2, len(box_features), figsize=(5 * len(box_features), 14))

for i, feat in enumerate(box_features):
    sample_box = df_sample_pd[[feat, "Label"]].dropna()
    sample_box = sample_box[sample_box["Label"].isin(top_classes_box)]

    ax_top = axes[0, i]
    sns.boxplot(data=sample_box, x="Label", y=feat, hue="Label",
                palette=COLOR_PALETTE, fliersize=1,
                legend=False, ax=ax_top)
    ax_top.set_title(f"{feat}\n(escala completa)", fontsize=10)
    ax_top.tick_params(axis="x", rotation=60, labelsize=7)
    ax_top.set_xlabel("")

    q_low, q_high = sample_box[feat].quantile(0.01), sample_box[feat].quantile(0.99)
    sample_clip = sample_box[(sample_box[feat] >= q_low) & (sample_box[feat] <= q_high)]
    ax_bot = axes[1, i]
    sns.boxplot(data=sample_clip, x="Label", y=feat, hue="Label",
                palette=COLOR_PALETTE, fliersize=1,
                legend=False, ax=ax_bot)
    ax_bot.set_title(f"{feat}\n(recortado P1–P99)", fontsize=10)
    ax_bot.tick_params(axis="x", rotation=60, labelsize=7)
    ax_bot.set_xlabel("")

fig.suptitle("Box Plots — Outliers y Distribución por Clase", fontsize=14, y=1.03)
fig.text(0.5, -0.01,
        "Fila superior: escala completa (outliers extremos visibles). "
        "Fila inferior: recorte P1–P99 (estructura central).",
        ha="center", fontsize=10, style="italic")
plt.tight_layout()
save_figure(fig, "07_box_plots")



📊 Gráfico 7: Box plots (doble perspectiva)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/07_box_plots.png


In [12]:
# === GRÁFICO 8: QQ-plots ===
# Se seleccionan 3 features representativas de los distintos perfiles
# distribucionales: (1) temporal, (2) con outliers negativos, (3) perfil distinto.
# El resto se documentan en el Anexo.
print("📊 Gráfico 8: QQ-plots (features representativas)")

qq_map = {"temporal": ["Flow Duration", "Fwd IAT Tot"],
        "negativos": ["Fwd IAT Min", "Flow IAT Min"],
        "distinto": ["Idle Max", "Idle Mean", "Fwd IAT Std"]}
qq_features = []
for role, candidates in qq_map.items():
    for c in candidates:
        if c in df_sample_pd.columns:
            qq_features.append(c)
            break
if len(qq_features) < 3:
    qq_features = top_features[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, feat in enumerate(qq_features[:3]):
    ax = axes[i]
    data = df_sample_pd[feat].dropna().values
    if len(data) > 5000:
        data = np.random.choice(data, 5000, replace=False)
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"QQ-Plot: {feat}", fontsize=11)

fig.suptitle("QQ-Plots — Comprobación de Normalidad (features representativas)",
            fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, "08_qqplots")
print("   📌 Patrón común: colas pesadas + zero-inflation → distribuciones fuertemente no-normales.")
print("  	QQ-plots completos disponibles en el Anexo.")

# QQ-plots adicionales → Anexo
qq_all = top_features[:6]
qq_annex = [f for f in qq_all if f not in qq_features]
if qq_annex:
    n_a = len(qq_annex)
    fig_a, axes_a = plt.subplots(1, n_a, figsize=(6 * n_a, 6))
    if n_a == 1:
        axes_a = [axes_a]
    for i, feat in enumerate(qq_annex):
        data = df_sample_pd[feat].dropna().values
        if len(data) > 5000:
            data = np.random.choice(data, 5000, replace=False)
        stats.probplot(data, dist="norm", plot=axes_a[i])
        axes_a[i].set_title(f"QQ-Plot: {feat}", fontsize=11)
    fig_a.suptitle("QQ-Plots — Features adicionales (Anexo)", fontsize=14, y=1.02)
    plt.tight_layout()
    save_annex_figure(fig_a, "A0_qqplots_adicionales")

📊 Gráfico 8: QQ-plots (features representativas)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/08_qqplots.png
   📌 Patrón común: colas pesadas + zero-inflation → distribuciones fuertemente no-normales.
  	QQ-plots completos disponibles en el Anexo.
  💾 Figura guardada (Anexo): /home/hpc/22231088student/tfg_ids/figures/anexo/A0_qqplots_adicionales.png


In [13]:
# === GRÁFICO 9: Ridge plots ===
# Se generan dos versiones: una para clases con distribución extendida
# y otra para clases concentradas cerca de 0, con eje reescalado.
print("📊 Gráfico 9: Ridge plots (separados por varianza de clase)")

ridge_feature = top_features[0]
ridge_high = [c for c in ["Benign", "DDoS attacks-LOIC-HTTP", "Infilteration",
                        "DoS attacks-GoldenEye"] if c in all_labels]
ridge_low = [c for c in ["DoS attacks-Hulk", "DDoS attack-HOIC", "SSH-Bruteforce",
                        "Bot", "FTP-BruteForce"] if c in all_labels]

def _plot_ridge(classes, feature, df_data, suffix, subtitle=""):
    """Genera ridge plot para un subconjunto de clases."""
    sample = df_data[[feature, "Label"]].dropna()
    sample = sample[sample["Label"].isin(classes)]
    if len(sample) == 0:
        print(f"  ⚠️ Sin datos para ridge {suffix}")
        return
    fig, axs = plt.subplots(len(classes), 1, figsize=(14, 2 * len(classes)), sharex=True)
    if len(classes) == 1:
        axs = [axs]
    for i, cls in enumerate(classes):
        ax = axs[i]
        d = sample[sample["Label"] == cls][feature]
        q1, q99 = d.quantile(0.01), d.quantile(0.99)
        d = d[(d >= q1) & (d <= q99)]
        if len(d) > 10:
            try:
                d.plot.kde(ax=ax, color=COLOR_PALETTE.get(cls, "#95a5a6"), linewidth=2)
                ax.fill_between(ax.lines[0].get_xdata(), ax.lines[0].get_ydata(),
                                alpha=0.3, color=COLOR_PALETTE.get(cls, "#95a5a6"))
            except Exception:
                ax.hist(d, bins=30, density=True, alpha=0.5,
                        color=COLOR_PALETTE.get(cls, "#95a5a6"))
        ax.set_ylabel("")
        ax.set_yticks([])
        ax.text(-0.01, 0.5, cls, transform=ax.transAxes, ha="right", va="center", fontsize=9)
        for spine in ["left", "right", "top"]:
            ax.spines[spine].set_visible(False)
    axs[-1].set_xlabel(feature)
    fig.suptitle(f"Ridge Plot — '{feature}' {subtitle}", fontsize=14, y=1.01)
    plt.tight_layout()
    save_figure(fig, f"09_ridge_plot_{suffix}")

_plot_ridge(ridge_high, ridge_feature, df_sample_pd,
            "alta_varianza", "(clases con distribución extendida)")
_plot_ridge(ridge_low, ridge_feature, df_sample_pd,
            "baja_varianza", "(clases concentradas — eje reescalado)")

📊 Gráfico 9: Ridge plots (separados por varianza de clase)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/09_ridge_plot_alta_varianza.png
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/09_ridge_plot_baja_varianza.png


---
## 2.3 — Análisis Bivariante

In [14]:
# === GRÁFICO 10: Heatmap de correlación ===
print("📊 Gráfico 10: Heatmap de correlación de Pearson")

df_numeric_pd = df_sample_pd[top_features].apply(pd.to_numeric, errors="coerce")
corr_matrix = df_numeric_pd.corr(method="pearson")

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Heatmap de Correlación de Pearson — Top Features", fontsize=16)
plt.tight_layout()
save_figure(fig, "10_heatmap_correlacion")

📊 Gráfico 10: Heatmap de correlación de Pearson
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/10_heatmap_correlacion.png


In [15]:
# === GRÁFICO 11: Scatter plots alta correlación → ANEXO ===
# Para datasets de alta cardinalidad (>16M registros) el hexbin (Gráfico 12)
# representa mejor la densidad. Los scatter se guardan como material complementario.
print("📊 Gráfico 11: Scatter plots de pares con alta correlación → Anexo")

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append(
                (corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
pairs_to_plot = high_corr_pairs[:4]

if pairs_to_plot:
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    scatter_sample = df_sample_pd.sample(n=min(5000, len(df_sample_pd)), random_state=SEED)
    top6_classes = label_counts["Label"].head(6).tolist()

    for i, (feat1, feat2, corr_val) in enumerate(pairs_to_plot):
        ax = axes[i]
        for cls in top6_classes:
            mask_cls = scatter_sample["Label"] == cls
            ax.scatter(scatter_sample.loc[mask_cls, feat1],
                    scatter_sample.loc[mask_cls, feat2],
                    alpha=0.4, s=10, label=cls,
                    color=COLOR_PALETTE.get(cls, "#95a5a6"))
        ax.set_xlabel(feat1, fontsize=10)
        ax.set_ylabel(feat2, fontsize=10)
        ax.set_title(f"r = {corr_val:.3f}", fontsize=12)
        ax.legend(fontsize=7, markerscale=2)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Scatter Plots — Pares con Alta Correlación (Anexo)", fontsize=16, y=1.02)
    plt.tight_layout()
    save_annex_figure(fig, "A1_scatter_alta_correlacion")

📊 Gráfico 11: Scatter plots de pares con alta correlación → Anexo
  💾 Figura guardada (Anexo): /home/hpc/22231088student/tfg_ids/figures/anexo/A1_scatter_alta_correlacion.png


In [16]:
# === GRÁFICO 12: Hexbin plots ===
# Se emplea hexbin en lugar de scatter dada la cardinalidad del dataset (>16M registros),
# donde la sobreposición de puntos impide visualizar la densidad real.
print("📊 Gráfico 12: Hexbin plots (densidad de puntos)")

if len(top_features) >= 4:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    feat_pairs_hex = [(top_features[0], top_features[1]),
                    (top_features[2], top_features[3])]

    for i, (f1, f2) in enumerate(feat_pairs_hex):
        ax = axes[i]
        data_hex = df_sample_pd[[f1, f2]].dropna()
        q1l, q1h = data_hex[f1].quantile(0.02), data_hex[f1].quantile(0.98)
        q2l, q2h = data_hex[f2].quantile(0.02), data_hex[f2].quantile(0.98)
        data_hex = data_hex[(data_hex[f1] >= q1l) & (data_hex[f1] <= q1h) &
                            (data_hex[f2] >= q2l) & (data_hex[f2] <= q2h)]
        hb = ax.hexbin(data_hex[f1], data_hex[f2], gridsize=40, cmap="YlOrRd", mincnt=1)
        ax.set_xlabel(f1)
        ax.set_ylabel(f2)
        ax.set_title(f"Hexbin: {f1} vs {f2}")
        plt.colorbar(hb, ax=ax, label="Conteo")

    fig.suptitle("Hexbin Plots — Densidad de Puntos", fontsize=14, y=1.03)
    plt.tight_layout()
    save_figure(fig, "12_hexbin_plots")

📊 Gráfico 12: Hexbin plots (densidad de puntos)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/12_hexbin_plots.png


In [25]:
# === GRÁFICO 13: Pair plot ===
# Muestreo estratificado: min(500, tamaño_clase) por clase para mantener
# representatividad sin saturar visualmente el gráfico.
print("📊 Gráfico 13: Pair plot (muestra estratificada)")

pair_features = top_features[:5]
top5_classes = label_counts["Label"].head(5).tolist()

pair_data = df_sample_pd[pair_features + ["Label"]].dropna()
pair_data = pair_data[pair_data["Label"].isin(top5_classes)]

# Muestreo estratificado sin usar groupby.apply (evita problemas de índice)
PAIR_N_PER_CLASS = 500
pair_sample_parts = []
for cls in top5_classes:
	cls_data = pair_data[pair_data["Label"] == cls]
	n_sample = min(PAIR_N_PER_CLASS, len(cls_data))
	pair_sample_parts.append(cls_data.sample(n=n_sample, random_state=SEED))
pair_sample = pd.concat(pair_sample_parts, ignore_index=True)

for feat in pair_features:
	q_low, q_high = pair_sample[feat].quantile(0.02), pair_sample[feat].quantile(0.98)
	pair_sample = pair_sample[(pair_sample[feat] >= q_low) & (pair_sample[feat] <= q_high)]

print(f"  Muestra estratificada: {len(pair_sample):,} registros")
for cls in top5_classes:
	print(f"	{cls}: {(pair_sample['Label'] == cls).sum()}")

palette_subset = {k: v for k, v in COLOR_PALETTE.items() if k in top5_classes}
g = sns.pairplot(pair_sample, hue="Label", palette=palette_subset,
             	plot_kws={"alpha": 0.4, "s": 15}, diag_kind="kde", height=2.5)
g.fig.suptitle(f"Pair Plot — Top 5 Features × Top 5 Clases\n"
           	f"(muestra estratificada: máx. {PAIR_N_PER_CLASS} por clase)",
           	y=1.03, fontsize=14)
save_figure(g.fig, "13_pair_plot")



📊 Gráfico 13: Pair plot (muestra estratificada)
  Muestra estratificada: 2,181 registros
	Benign: 389
	DDoS attack-HOIC: 500
	DDoS attacks-LOIC-HTTP: 307
	DoS attacks-Hulk: 499
	Bot: 486
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/13_pair_plot.png


In [26]:
# === GRÁFICO 14: Clustermap ===
print("📊 Gráfico 14: Clustermap (correlación jerárquica)")

clustermap_features = sorted(variances, key=variances.get, reverse=True)[:20]
clustermap_features = [f for f in clustermap_features if f in df_sample_pd.columns]

df_cluster_pd = df_sample_pd[clustermap_features].apply(pd.to_numeric, errors="coerce")
corr_cluster = df_cluster_pd.corr()

g = sns.clustermap(corr_cluster, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                figsize=(14, 14), linewidths=0.5, annot=False, dendrogram_ratio=0.15)
g.fig.suptitle("Clustermap — Correlación Jerárquica de Features", fontsize=16, y=1.02)
save_figure(g.fig, "14_clustermap")

print("\n📌 Clusters principales identificados por el dendrograma:")
print("   Cluster A (correlación positiva entre sí, negativa con B):")
print(" 	→ Fwd IAT Std, Flow IAT Std, Idle Mean, Idle Max, Flow IAT Max, Fwd IAT Max")
print("   Cluster B (correlación positiva entre sí, negativa con A):")
print(" 	→ Fwd IAT Min, Flow IAT Min, Flow Duration, Fwd IAT Tot")
print("   ⚠️ Alta multicolinealidad dentro de cada cluster → candidatas a reducción de features.")

📊 Gráfico 14: Clustermap (correlación jerárquica)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/14_clustermap.png

📌 Clusters principales identificados por el dendrograma:
   Cluster A (correlación positiva entre sí, negativa con B):
 	→ Fwd IAT Std, Flow IAT Std, Idle Mean, Idle Max, Flow IAT Max, Fwd IAT Max
   Cluster B (correlación positiva entre sí, negativa con A):
 	→ Fwd IAT Min, Flow IAT Min, Flow Duration, Fwd IAT Tot
   ⚠️ Alta multicolinealidad dentro de cada cluster → candidatas a reducción de features.


---
## 2.4 — Análisis Multivariante y Reducción Dimensional

In [27]:
# === Helper: elipse de confianza para PCA ===
def draw_confidence_ellipse(x, y, ax, n_std=2.0, facecolor="none", edgecolor="black", **kwargs):
    """Dibuja una elipse de confianza al n_std desviaciones estándar."""
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    if np.any(np.isnan(cov)) or np.any(np.isinf(cov)):
        return
    pearson = cov[0, 1] / (np.sqrt(cov[0, 0] * cov[1, 1]) + 1e-12)
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2,
                    facecolor=facecolor, edgecolor=edgecolor, **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    mean_x, mean_y = np.mean(x), np.mean(y)
    transf = (transforms.Affine2D()
            .rotate_deg(45)
            .scale(scale_x, scale_y)
            .translate(mean_x, mean_y))
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)

In [28]:
# === PREPARAR DATOS PARA PCA y t-SNE ===
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

DR_SAMPLE_SIZE = min(50_000, total_rows)
dr_fraction = DR_SAMPLE_SIZE / total_rows
df_dr_pd = df.select(numeric_cols + ["Label"]).sample(False, dr_fraction, seed=SEED).toPandas()
df_dr_pd = df_dr_pd.replace([np.inf, -np.inf], np.nan).dropna()

X_dr = df_dr_pd[numeric_cols].values
y_dr = df_dr_pd["Label"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dr)
print(f"📊 Datos para reducción dimensional: {X_scaled.shape[0]:,} × {X_scaled.shape[1]}")

26/05/03 21:42:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

📊 Datos para reducción dimensional: 24,134 × 79


In [29]:
# === GRÁFICO 15: PCA 2D ===
# Se añaden elipses de confianza al 95% por clase para formalizar
# visualmente la separabilidad de los clusters.
print("📊 Gráfico 15: PCA 2D con elipses de confianza al 95%")

pca_2d = PCA(n_components=2, random_state=SEED)
X_pca_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(14, 10))
top_classes_pca = label_counts["Label"].head(10).tolist()

for cls in top_classes_pca:
    mask_cls = y_dr == cls
    x_cls = X_pca_2d[mask_cls, 0]
    y_cls = X_pca_2d[mask_cls, 1]
    color = COLOR_PALETTE.get(cls, "#95a5a6")
    ax.scatter(x_cls, y_cls, alpha=0.3, s=8, label=cls, color=color)
    # Elipse de confianza al 95% (solo si hay suficientes puntos)
    if mask_cls.sum() > 50:
        draw_confidence_ellipse(x_cls, y_cls, ax, n_std=2.0,
                                edgecolor=color, linewidth=2, linestyle="--", alpha=0.8)

ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza)")
ax.set_title("PCA 2D — Separabilidad de Clases (con elipses de confianza al 95%)")
ax.legend(fontsize=9, markerscale=3, bbox_to_anchor=(1.05, 1))
plt.tight_layout()
save_figure(fig, "15_pca_2d")

📊 Gráfico 15: PCA 2D con elipses de confianza al 95%
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/15_pca_2d.png


### 📌 Nota: elección de t-SNE 3D frente a PCA 3D

**PCA** es una técnica **lineal** que preserva la estructura global (varianza), pero cuando
las clases no son linealmente separables en pocas componentes, la proyección colapsa los
clusters y dificulta su visualización.

**t-SNE** es una técnica **no lineal** que preserva la estructura **local** (vecindarios),
revelando agrupaciones que PCA no captura en 2–3 dimensiones
(van der Maaten & Hinton, 2008). Estudios recientes en detección de intrusiones confirman
que t-SNE es más efectivo que PCA para visualizar la separación entre tipos de tráfico de
red, dado el carácter no lineal de sus relaciones (Ghani et al., 2024).

⚠️ **Limitación**: t-SNE es estocástico y las distancias absolutas entre clusters no son
interpretables — solo la estructura de agrupación es significativa.



In [71]:
# === GRÁFICO 16: t-SNE 3D — Exploración de Clusters (Top 6 clases) ===
# Se usa t-SNE 3D en lugar de PCA 3D porque las top features por varianza
# están altamente correlacionadas (PC1=72%), colapsando la proyección PCA
# en una línea. t-SNE, al ser no lineal, revela la estructura de clusters.
from sklearn.manifold import TSNE as _TSNE3D
from sklearn.preprocessing import StandardScaler as _StdSc3D

print("📊 Gráfico 16: t-SNE 3D — Exploración de Clusters (Top 6 clases)")

# --- Top 6 clases ---
top6_classes_3d = label_counts["Label"].head(6).tolist()

# --- Datos desde df_sample_pd ---
tsne3d_data = df_sample_pd[top_features + ["Label"]].copy()
tsne3d_data = tsne3d_data[tsne3d_data["Label"].isin(top6_classes_3d)]
tsne3d_data = tsne3d_data.replace([np.inf, -np.inf], np.nan).dropna()

# --- Muestreo ESTRATIFICADO EQUITATIVO ---
N_3D = 10_000
n_per_class = N_3D // len(top6_classes_3d)

parts_3d = []
for cls in top6_classes_3d:
    cls_data = tsne3d_data[tsne3d_data["Label"] == cls]
    n_take = min(n_per_class, len(cls_data))
    parts_3d.append(cls_data.sample(n=n_take, random_state=SEED))
    print(f"   {cls}: {n_take:,} muestras seleccionadas (de {len(cls_data):,} disponibles)")

tsne3d_sample = pd.concat(parts_3d, ignore_index=True)

# --- StandardScaler ---
scaler_3d = _StdSc3D()
X_3d_scaled = scaler_3d.fit_transform(tsne3d_sample[top_features].values)

# --- t-SNE 3D ---
TSNE_PERP_3D = 30
print(f"\n   Ejecutando t-SNE 3D (perplexity={TSNE_PERP_3D}) sobre {len(X_3d_scaled):,} muestras...")
tsne_3d = _TSNE3D(n_components=3, random_state=SEED, perplexity=TSNE_PERP_3D, max_iter=1000)
X_tsne_3d = tsne_3d.fit_transform(X_3d_scaled)

# --- DataFrame para Plotly ---
df_tsne3d = pd.DataFrame({
    "Dim1": X_tsne_3d[:, 0],
    "Dim2": X_tsne_3d[:, 1],
    "Dim3": X_tsne_3d[:, 2],
    "Label": tsne3d_sample["Label"].values,
})

n_total = len(df_tsne3d)
print(f"   t-SNE completado: {n_total:,} muestras en 3D")

# --- Gráfico 3D (todas las clases con colores iguales) ---
fig_3d = px.scatter_3d(
	df_tsne3d,
	x="Dim1", y="Dim2", z="Dim3",
	color="Label",
	opacity=0.8,
	color_discrete_sequence=px.colors.qualitative.Bold,
	category_orders={"Label": top6_classes_3d},
	title=(
    	f"t-SNE 3D — Exploración de Clusters (Top 6 clases)<br>"
    	f"<sup>N={n_total:,} muestras (equitativo) | "
    	f"perplexity={TSNE_PERP_3D}</sup>"
	),
)
fig_3d.update_traces(marker=dict(size=3))
fig_3d.update_layout(
	legend_title_text="Clase",
	margin=dict(l=0, r=0, t=80, b=0),
	width=950, height=750,
	scene=dict(
    	xaxis_title="t-SNE Dim. 1",
    	yaxis_title="t-SNE Dim. 2",
    	zaxis_title="t-SNE Dim. 3",
	),
)
fig_3d.show()


fig_3d.write_html(os.path.join(FIGURES_PATH, "16_tsne_3d.html"))
print(f"   💾 Figura guardada: {os.path.join(FIGURES_PATH, '16_tsne_3d.html')}")
try:
    fig_3d.write_image(os.path.join(FIGURES_PATH, "16_tsne_3d.png"), scale=2)
    print(f"   💾 Figura guardada: {os.path.join(FIGURES_PATH, '16_tsne_3d.png')}")
except Exception:
    print("   ⚠️ PNG no generado (kaleido no disponible) — incluir screenshot en el PDF")

print("\n   ⚠️ NOTA: t-SNE es estocástico. Las distancias absolutas entre clusters")
print("   NO son interpretables geométricamente — solo la estructura de agrupación.")



📊 Gráfico 16: t-SNE 3D — Exploración de Clusters (Top 6 clases)
   Benign: 1,666 muestras seleccionadas (de 82,968 disponibles)
   DDoS attack-HOIC: 1,666 muestras seleccionadas (de 4,275 disponibles)
   DDoS attacks-LOIC-HTTP: 1,666 muestras seleccionadas (de 3,507 disponibles)
   DoS attacks-Hulk: 1,666 muestras seleccionadas (de 2,809 disponibles)
   Bot: 1,666 muestras seleccionadas (de 1,769 disponibles)
   FTP-BruteForce: 1,193 muestras seleccionadas (de 1,193 disponibles)

   Ejecutando t-SNE 3D (perplexity=30) sobre 9,523 muestras...
   t-SNE completado: 9,523 muestras en 3D


   💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/16_tsne_3d.html
   ⚠️ PNG no generado (kaleido no disponible) — incluir screenshot en el PDF

   ⚠️ NOTA: t-SNE es estocástico. Las distancias absolutas entre clusters
   NO son interpretables geométricamente — solo la estructura de agrupación.


In [31]:
# === GRÁFICO 17: Scree plot ===
print("📊 Gráfico 17: Scree plot")

pca_full = PCA(random_state=SEED)
pca_full.fit(X_scaled)
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(range(1, len(explained_var) + 1), explained_var, alpha=0.6,
   	color="#3498db", label="Varianza individual")
ax.plot(range(1, len(cumulative_var) + 1), cumulative_var, "r-o",
    	markersize=4, label="Varianza acumulada")
ax.axhline(y=0.95, color="#e74c3c", linestyle="--", linewidth=1.5, label="Umbral 95%")

n_components_95 = np.argmax(cumulative_var >= 0.95) + 1
ax.axvline(x=n_components_95, color="#2ecc71", linestyle="--", linewidth=1.5,
       	label=f"{n_components_95} componentes → 95%")

ax.set_xlabel("Componente Principal")
ax.set_ylabel("Varianza Explicada")
ax.set_title("Scree Plot — Varianza Explicada por Componente PCA")
ax.legend(fontsize=11)
ax.set_xlim(0, min(50, len(explained_var)))
plt.tight_layout()
save_figure(fig, "17_scree_plot")
print(f"  📌 Se necesitan {n_components_95} componentes para el 95% de varianza.")

📊 Gráfico 17: Scree plot
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/17_scree_plot.png
  📌 Se necesitan 21 componentes para el 95% de varianza.


In [32]:
# === GRÁFICO 18: t-SNE 2D ===
# t-SNE es una técnica estocástica de reducción de dimensionalidad no lineal.
# Los resultados no son reproducibles entre ejecuciones salvo fijación de random_state
# y las distancias absolutas entre clusters no son interpretables geométricamente.
print("📊 Gráfico 18: t-SNE 2D (muestra representativa)")

TSNE_PERPLEXITY = 30
TSNE_SAMPLE = min(10_000, len(X_scaled))
idx_tsne = np.random.choice(len(X_scaled), TSNE_SAMPLE, replace=False)
X_tsne_input = X_scaled[idx_tsne]
y_tsne = y_dr[idx_tsne]

print(f"  Ejecutando t-SNE (perplexity={TSNE_PERPLEXITY}) sobre {TSNE_SAMPLE:,} muestras...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=TSNE_PERPLEXITY, max_iter=1000)
X_tsne_2d = tsne.fit_transform(X_tsne_input)

fig, ax = plt.subplots(figsize=(14, 10))
top_classes_tsne = label_counts["Label"].head(10).tolist()

for cls in top_classes_tsne:
    mask_cls = y_tsne == cls
    ax.scatter(X_tsne_2d[mask_cls, 0], X_tsne_2d[mask_cls, 1],
            alpha=0.4, s=10, label=cls, color=COLOR_PALETTE.get(cls, "#95a5a6"))

ax.set_xlabel("t-SNE Dim. 1")
ax.set_ylabel("t-SNE Dim. 2")
ax.set_title(f"t-SNE 2D — Agrupación No Lineal de Clases\n"
            f"(perplexity={TSNE_PERPLEXITY}, n={TSNE_SAMPLE:,}, random_state={SEED})")
ax.legend(fontsize=9, markerscale=3, bbox_to_anchor=(1.05, 1))
plt.tight_layout()
save_figure(fig, "18_tsne_2d")
print("  ⚠️ NOTA: t-SNE es estocástico. Las distancias entre clusters NO son interpretables.")

📊 Gráfico 18: t-SNE 2D (muestra representativa)
  Ejecutando t-SNE (perplexity=30) sobre 10,000 muestras...
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/18_tsne_2d.png
  ⚠️ NOTA: t-SNE es estocástico. Las distancias entre clusters NO son interpretables.


In [60]:
# === GRÁFICO 19: Parallel Coordinates — Perfil de Features por Clase ===
# Se usa RobustScaler (resistente a outliers) en lugar de MinMax para evitar
# compresión extrema de valores. Colores categóricos con leyenda explícita.
from sklearn.preprocessing import RobustScaler as _RobustScalerPC

print("📊 Gráfico 19: Parallel Coordinates (RobustScaler + colores categóricos)")

parallel_features = top_features[:8]
top_classes_par = label_counts["Label"].head(6).tolist()

parallel_sample = df_sample_pd[parallel_features + ["Label"]].dropna()
parallel_sample = parallel_sample[parallel_sample["Label"].isin(top_classes_par)]

if len(parallel_sample) > 3000:
	parallel_sample = parallel_sample.sample(3000, random_state=SEED)

# --- RobustScaler + clip [-5, 5] para legibilidad ---
scaler_par = _RobustScalerPC()
X_par_scaled = scaler_par.fit_transform(parallel_sample[parallel_features].values)
X_par_clipped = np.clip(X_par_scaled, -5, 5)

# --- Codificación categórica de clases ---
class_names_par = sorted(parallel_sample["Label"].unique())
class_to_id_par = {name: i for i, name in enumerate(class_names_par)}
parallel_sample = parallel_sample.copy()
parallel_sample["class_id"] = parallel_sample["Label"].map(class_to_id_par)

# --- Escala de color categórica ---
colors_par = px.colors.qualitative.Set2[:len(class_names_par)]
colorscale_par = []
for i, color in enumerate(colors_par):
	low = i / len(class_names_par)
	high = (i + 1) / len(class_names_par)
	colorscale_par.append([low, color])
	colorscale_par.append([high, color])

# --- Dimensiones (ejes) ---
dimensions_par = []
for i, col in enumerate(parallel_features):
	dimensions_par.append(dict(
    	range=[float(X_par_clipped[:, i].min()), float(X_par_clipped[:, i].max())],
    	label=col,
    	values=X_par_clipped[:, i],
	))

# --- Gráfico Parcoords ---
fig_par = go.Figure(
	data=go.Parcoords(
    	line=dict(
        	color=parallel_sample["class_id"].values,
        	colorscale=colorscale_par,
        	showscale=False,
        	cmin=0,
        	cmax=len(class_names_par) - 1,
    	),
    	dimensions=dimensions_par,
	)
)

# --- Leyenda manual (Parcoords no soporta leyenda nativa en Plotly) ---
for i, name in enumerate(class_names_par):
	fig_par.add_trace(go.Scatter(
    	x=[None], y=[None],
    	mode="markers",
    	marker=dict(size=10, color=colors_par[i]),
    	name=name,
    	showlegend=True,
	))

fig_par.update_layout(
	title=dict(
    	text=(
        	f"Parallel Coordinates — Perfil de Features por Clase (RobustScaler)<br>"
        	f"<sup>Top 6 clases | N={len(parallel_sample):,} muestras | Clipped [-5, 5]</sup>"
    	),
    	y=0.98,
    	x=0.5,
    	xanchor="center",
    	yanchor="top",
	),
	width=1200, height=700,
	margin=dict(l=80, r=80, t=120, b=40),
)
fig_par.show()
fig_par.write_html(os.path.join(FIGURES_PATH, "19_parallel_coordinates.html"))
print(f"   💾 Figura guardada: {os.path.join(FIGURES_PATH, '19_parallel_coordinates.html')}")
try:
	fig_par.write_image(os.path.join(FIGURES_PATH, "19_parallel_coordinates.png"), scale=2)
	print(f"   💾 Figura guardada: {os.path.join(FIGURES_PATH, '19_parallel_coordinates.png')}")
except Exception:
	print("   ⚠️ PNG no generado (kaleido no disponible) — incluir screenshot en el PDF")

📊 Gráfico 19: Parallel Coordinates (RobustScaler + colores categóricos)


   💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/19_parallel_coordinates.html
   ⚠️ PNG no generado (kaleido no disponible) — incluir screenshot en el PDF


---
## 2.5 — Análisis Temporal

In [34]:
# === PARSEO ROBUSTO DE TIMESTAMPS ===
# Algunos registros tienen timestamps que se parsean como epoch (año 1970).
# Se recupera la fecha real a partir del nombre del fichero fuente (source_file),
# que contiene la fecha en formato "dd-MM-yyyy".
print("🔧 Parseando timestamps con verificación de calidad...")

df_temporal = df.withColumn("date_parsed", F.to_date(F.col("Timestamp"), "dd/MM/yyyy HH:mm:ss"))

if "source_file" in df.columns:
    print("  ✅ Columna 'source_file' encontrada → recuperando fechas desde nombre de archivo")
    df_temporal = df_temporal.withColumn(
        "date_from_file",
        F.to_date(
            F.regexp_extract(F.col("source_file"), r"(\d{2}-\d{2}-\d{4})", 1),
            "dd-MM-yyyy"
        )
    )
    df_temporal = df_temporal.withColumn(
        "date",
        F.when(F.year(F.col("date_parsed")) >= 2000, F.col("date_parsed"))
        .otherwise(F.col("date_from_file"))
    )
    n_recovered = df_temporal.filter(
        (F.year(F.col("date_parsed")) < 2000) & (F.col("date_from_file").isNotNull())
    ).count()
    print(f"  📌 {n_recovered:,} registros con timestamp inválido recuperados desde source_file")
else:
    print("  ⚠️ Columna 'source_file' no encontrada → filtrando registros con año < 2000")
    df_temporal = df_temporal.withColumn("date", F.col("date_parsed"))
    n_invalid = df_temporal.filter(F.year(F.col("date")) < 2000).count()
    df_temporal = df_temporal.filter(F.year(F.col("date")) >= 2000)
    print(f"  📌 {n_invalid:,} registros con fecha inválida excluidos")

df_temporal = df_temporal.filter(F.col("date").isNotNull())

🔧 Parseando timestamps con verificación de calidad...
  ⚠️ Columna 'source_file' no encontrada → filtrando registros con año < 2000


[Stage 251:====================================================>(123 + 1) / 124]

  📌 14 registros con fecha inválida excluidos


In [35]:
# === GRÁFICO 20: Volumen de tráfico por día ===
print("📊 Gráfico 20: Volumen de tráfico por día")

traffic_volume = df_temporal.groupBy("date").count().orderBy("date").toPandas()

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(traffic_volume["date"].astype(str), traffic_volume["count"],
        "b-o", markersize=8, linewidth=2)
ax.fill_between(range(len(traffic_volume)), traffic_volume["count"],
                alpha=0.15, color="#3498db")
ax.set_xlabel("Fecha")
ax.set_ylabel("Número de Registros")
ax.set_title("Volumen de Tráfico por Día de Captura")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_figure(fig, "20_volumen_trafico_temporal")

📊 Gráfico 20: Volumen de tráfico por día


  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/20_volumen_trafico_temporal.png


In [36]:
# === GRÁFICO 21: Stacked area chart ===
# Se generan dos versiones: absoluta y 100% proporcional, para comparar
# tanto el volumen como la composición de ataques por día.
print("📊 Gráfico 21: Stacked area chart (absoluto + proporcional)")

attack_by_date = df_temporal.groupBy("date", "Label").count().toPandas()
pivot_attacks = attack_by_date.pivot_table(
    index="date", columns="Label", values="count", fill_value=0).sort_index()
pivot_attacks.index = pivot_attacks.index.astype(str)

# 21a: Valores absolutos
fig, ax = plt.subplots(figsize=(18, 8))
colors_stacked = [COLOR_PALETTE.get(col, "#95a5a6") for col in pivot_attacks.columns]
pivot_attacks.plot.area(ax=ax, stacked=True, color=colors_stacked, alpha=0.8)
ax.set_xlabel("Fecha")
ax.set_ylabel("Número de Registros")
ax.set_title("Composición de Ataques por Día — Valores Absolutos")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_figure(fig, "21a_stacked_area_absoluto")

# 21b: Proporciones relativas (100% stacked)
pivot_pct = pivot_attacks.div(pivot_attacks.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(18, 8))
pivot_pct.plot.area(ax=ax, stacked=True, color=colors_stacked, alpha=0.8)
ax.set_xlabel("Fecha")
ax.set_ylabel("Proporción (%)")
ax.set_title("Composición de Ataques por Día — Proporciones Relativas (100% Stacked)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax.tick_params(axis="x", rotation=45)
ax.set_ylim(0, 100)
plt.tight_layout()
save_figure(fig, "21b_stacked_area_proporcional")

📊 Gráfico 21: Stacked area chart (absoluto + proporcional)


  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/21a_stacked_area_absoluto.png
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/21b_stacked_area_proporcional.png


In [37]:
# === GRÁFICO 22: Heatmap temporal ===
print("📊 Gráfico 22: Heatmap temporal (fechas × clases)")

heatmap_data = attack_by_date.pivot_table(
    index="date", columns="Label", values="count", fill_value=0).sort_index()
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)
heatmap_norm.index = heatmap_norm.index.astype(str)

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(heatmap_norm.T, cmap="YlOrRd", annot=False, linewidths=0.5, ax=ax)
ax.set_xlabel("Fecha")
ax.set_ylabel("Tipo de Ataque")
ax.set_title("Heatmap — Proporción de Cada Ataque por Día")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_figure(fig, "22_heatmap_temporal")

📊 Gráfico 22: Heatmap temporal (fechas × clases)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/22_heatmap_temporal.png


---
## 2.6 — Perfiles de Ataque

In [38]:
# === GRÁFICO 23: Radar charts ===
# Se usan estilos de línea y marcadores diferenciados para evitar
# solapamiento visual entre clases con perfiles similares.
print("📊 Gráfico 23: Radar charts (perfil medio por ataque)")

radar_features = top_features[:8]
top_classes_radar = label_counts["Label"].head(6).tolist()
means_by_class = df_sample_pd.groupby("Label")[radar_features].mean().loc[top_classes_radar]

for col in radar_features:
    col_min, col_max = means_by_class[col].min(), means_by_class[col].max()
    if col_max > col_min:
        means_by_class[col] = (means_by_class[col] - col_min) / (col_max - col_min)

N = len(radar_features)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

line_styles = ["-", "--", "-.", ":", "-", "--"]
markers = ["o", "s", "^", "D", "v", "*"]

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))
for idx, cls in enumerate(top_classes_radar):
    values = means_by_class.loc[cls].values.tolist()
    values += values[:1]
    color = COLOR_PALETTE.get(cls, "#95a5a6")
    ax.plot(angles, values, linestyle=line_styles[idx % len(line_styles)],
            marker=markers[idx % len(markers)], markersize=6,
            linewidth=2, label=cls, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=9)
ax.set_title("Radar Chart — Perfil Medio por Tipo de Ataque", fontsize=16, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
save_figure(fig, "23_radar_chart")

📊 Gráfico 23: Radar charts (perfil medio por ataque)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/23_radar_chart.png


In [39]:
# === GRÁFICO 24: Grouped bar chart ===
# Se usa escala logarítmica simétrica (symlog) para poder visualizar
# simultáneamente clases con medias de magnitudes muy diferentes.
print("📊 Gráfico 24: Grouped bar chart (escala symlog)")

bar_features = top_features[:5]
top_classes_bar = label_counts["Label"].head(6).tolist()
means_bar = df_sample_pd.groupby("Label")[bar_features].mean().loc[top_classes_bar]

fig, ax = plt.subplots(figsize=(16, 8))
means_bar.plot(kind="bar", ax=ax, color=["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"])
ax.set_title("Media de Features Principales por Clase de Ataque (escala symlog)")
ax.set_xlabel("Clase de Ataque")
ax.set_ylabel("Valor Medio (escala symlog)")
ax.set_yscale("symlog")
ax.legend(title="Feature", bbox_to_anchor=(1.05, 1))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_figure(fig, "24_grouped_bar_medias")

# Tabla numérica con media ± std
print("\n📋 Tabla numérica: Media ± Std por clase y feature:")
stds_bar = df_sample_pd.groupby("Label")[bar_features].std().loc[top_classes_bar]
for cls in top_classes_bar:
    print(f"\n  {cls}:")
    for feat in bar_features:
        m = means_bar.loc[cls, feat]
        s = stds_bar.loc[cls, feat]
        print(f"	{feat:<25s}  {m:>15.2f} ± {s:>15.2f}")

📊 Gráfico 24: Grouped bar chart (escala symlog)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/24_grouped_bar_medias.png

📋 Tabla numérica: Media ± Std por clase y feature:

  Benign:
	Fwd IAT Min                    -7931792.58 ±   3184687722.91
	Flow IAT Min                   -8003254.64 ±   3184687065.96
	Flow IAT Max                   12233446.06 ±   1565349576.49
	Fwd IAT Max                    12036129.46 ±   1565349008.36
	Idle Max                       10809546.44 ±   1565341423.72

  DDoS attack-HOIC:
	Fwd IAT Min                        6986.64 ±         8976.32
	Flow IAT Min                       6909.41 ±         9034.25
	Flow IAT Max                       9220.36 ±         8716.20
	Fwd IAT Max                        8997.91 ±         8550.99
	Idle Max                              0.00 ±            0.00

  DDoS attacks-LOIC-HTTP:
	Fwd IAT Min                    10701394.44 ±     15404973.39
	Flow IAT Min                   10701365.50 ±     15404993.50
	Flow IA

### 📌 Nota sobre valores negativos en features IAT (Benign)

Se observa que la clase **Benign** presenta valores medios **negativos** en `Fwd IAT Min` y
`Flow IAT Min` (del orden de -10⁶ μs). Esto es un **artefacto del dataset original
(CSE-CIC-IDS 2018)**, no un error del pipeline de procesamiento.

**Causa probable:** el generador de features *CICFlowMeter* calcula los inter-arrival times
(IAT) como diferencias entre timestamps consecutivos de paquetes dentro de un flujo. Cuando
los timestamps no están perfectamente ordenados —algo que puede ocurrir por microsegundos
de desfase en la captura, reordenamiento en el buffer de red, o flujos con un solo paquete
donde el IAT no está definido y se asigna un valor por defecto— el resultado puede ser
negativo.

**Impacto en el modelado:**
- Estos valores **no se eliminan**, ya que forman parte de la distribución real de la clase
  Benign y pueden aportar información discriminativa (ninguna clase de ataque presenta este
  patrón).
- En el preprocesado (NB3) se aplicará **RobustScaler**, resistente a estos valores extremos,
  minimizando su efecto sin perder la señal.
- Alternativamente podrían clipearse a 0, pero se opta por conservarlos para no introducir
  sesgo artificial.



In [41]:
# === GRÁFICO 25: Strip plots ===
# Winsorizado al P1–P99 para evitar que outliers extremos compriman
# toda la información útil. Se anota el número de outliers excluidos.
print("📊 Gráfico 25: Strip plots (winsorizado P1–P99)")

strip_feature = top_features[1]
top_classes_strip = label_counts["Label"].head(6).tolist()
strip_data = df_sample_pd[[strip_feature, "Label"]].dropna()
strip_data = strip_data[strip_data["Label"].isin(top_classes_strip)]

n_total = len(strip_data)
q_low = strip_data[strip_feature].quantile(0.01)
q_high = strip_data[strip_feature].quantile(0.99)
strip_clipped = strip_data[(strip_data[strip_feature] >= q_low) &
                        	(strip_data[strip_feature] <= q_high)]
n_excluded = n_total - len(strip_clipped)

if len(strip_clipped) > 3000:
	strip_clipped = strip_clipped.sample(3000, random_state=SEED)

fig, ax = plt.subplots(figsize=(14, 8))
sns.stripplot(data=strip_clipped, x="Label", y=strip_feature, hue="Label",
          	palette=COLOR_PALETTE, alpha=0.4, size=3, jitter=True,
          	legend=False, ax=ax)
ax.set_title(f"Strip Plot — '{strip_feature}' por Clase (P1–P99)\n"
         	f"({n_excluded:,} outliers excluidos, {n_excluded/n_total*100:.1f}%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_figure(fig, "25_strip_plot")



📊 Gráfico 25: Strip plots (winsorizado P1–P99)
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/25_strip_plot.png


---
## 2.7 — Calidad de Datos

In [42]:
# === GRÁFICO 26: Pipeline de normalización ===
print("📊 Gráfico 26: Impacto del pipeline de normalización de labels")

norm_data = pd.DataFrame({
    "Categoría": ["Labels ya válidos", "Recuperados (case-insensitive)", "Irrecuperables (cabeceras)"],
    "Filas": [15_545_201, 687_742, 59]
})

fig, ax = plt.subplots(figsize=(14, 7))
colors_norm = ["#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.barh(norm_data["Categoría"], norm_data["Filas"], color=colors_norm)
ax.set_xlabel("Número de Filas")
ax.set_title("Pipeline de Normalización de Labels — NB1")
ax.set_xscale("log")
for bar in bars:
    width = bar.get_width()
    ax.text(width * 1.1, bar.get_y() + bar.get_height() / 2,
            f"{int(width):,}", ha="left", va="center", fontsize=12, fontweight="bold")
plt.tight_layout()
save_figure(fig, "26_normalizacion_labels")

📊 Gráfico 26: Impacto del pipeline de normalización de labels
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/26_normalizacion_labels.png


In [44]:
# === GRÁFICO 27: Auditoría de calidad por archivo ===
print("📊 Gráfico 27: Labels válidos vs. inválidos por archivo")

audit_csv_path = os.path.join(FIGURES_PATH, "auditoria_calidad_por_archivo.csv")
if os.path.exists(audit_csv_path):
    audit_data = pd.read_csv(audit_csv_path)

    fig, ax = plt.subplots(figsize=(16, 8))
    x = range(len(audit_data))
    short_names = [name.split("_")[0] for name in audit_data["source_file"]]

    ax.bar(x, audit_data["labels_validos"], label="Labels Válidos", color="#2ecc71")
    ax.bar(x, audit_data["labels_invalidos"],
        bottom=audit_data["labels_validos"],
        label="Labels Inválidos", color="#e74c3c")
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha="right")
    ax.set_xlabel("Archivo Fuente")
    ax.set_ylabel("Número de Registros")
    ax.set_title("Labels Válidos vs. Inválidos por Archivo (Post-Normalización)")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, "27_stacked_bar_calidad_archivos")
else:
    print("  ⚠️ Archivo de auditoría no encontrado. Ejecutar NB1 primero.")

📊 Gráfico 27: Labels válidos vs. inválidos por archivo
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/27_stacked_bar_calidad_archivos.png


In [45]:
# === GRÁFICO 28: Auditoría de nulos e infinitos ===
# Se usa missingno.matrix() como estándar profesional para visualizar
# patrones de datos faltantes. Se complementa con tabla de auditoría.
print("📊 Gráfico 28: Auditoría de valores nulos e infinitos")

# Parte A: Visualización
if HAS_MISSINGNO:
    fig_msno = msno.matrix(df_sample_pd.sample(min(1000, len(df_sample_pd)), random_state=SEED),
                        figsize=(18, 8), sparkline=True, fontsize=8)
    fig_msno_fig = fig_msno.get_figure()
    fig_msno_fig.suptitle("Matriz de Datos Faltantes (missingno)", fontsize=14, y=1.02)
    save_figure(fig_msno_fig, "28_missingno_matrix")
else:
    null_matrix = df_sample_pd.isnull().astype(int)
    null_sample = null_matrix.sample(min(1000, len(null_matrix)), random_state=SEED)
    fig, ax = plt.subplots(figsize=(18, 8))
    sns.heatmap(null_sample.T, cmap="YlOrRd", vmin=0, vmax=1,
                cbar_kws={"label": "Nulo (1) / No nulo (0)"},
                yticklabels=True, xticklabels=False, ax=ax)
    ax.set_xlabel("Registros (muestreados)")
    ax.set_ylabel("Features")
    ax.set_title("Heatmap de Valores Nulos — Patrón de Datos Faltantes")
    plt.tight_layout()
    save_figure(fig, "28_heatmap_nulos")

# Parte B: Tabla de auditoría nulos + infinitos
print("\n📋 Auditoría de calidad — Nulos e Infinitos por Feature:")
print(f"   {'Feature':<30s} {'N_null':>8} {'%_null':>8} {'N_inf':>8} {'%_inf':>8}")
print("   " + "-" * 66)
has_issues = False
for col in df_sample_pd.columns:
    if col == "Label":
        continue
    n_null = df_sample_pd[col].isnull().sum()
    pct_null = n_null / len(df_sample_pd) * 100
    try:
        n_inf = np.isinf(df_sample_pd[col].astype(float)).sum()
    except (ValueError, TypeError):
        n_inf = 0
    pct_inf = n_inf / len(df_sample_pd) * 100
    if n_null > 0 or n_inf > 0:
        has_issues = True
        print(f"   {col:<30s} {n_null:>8,} {pct_null:>7.2f}% {n_inf:>8,} {pct_inf:>7.2f}%")

if not has_issues:
    print("   ✅ Todas las features tienen 0 nulos y 0 infinitos.")

📊 Gráfico 28: Auditoría de valores nulos e infinitos
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/28_missingno_matrix.png

📋 Auditoría de calidad — Nulos e Infinitos por Feature:
   Feature                          N_null   %_null    N_inf    %_inf
   ------------------------------------------------------------------
   ✅ Todas las features tienen 0 nulos y 0 infinitos.


In [46]:
# === GRÁFICO 29: Porcentaje de nulos por columna ===
print("📊 Gráfico 29: Porcentaje de nulos por columna")

null_pct = (df_sample_pd.isnull().sum() / len(df_sample_pd) * 100).sort_values(ascending=False)
null_pct_nonzero = null_pct[null_pct > 0]

if len(null_pct_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(14, max(6, len(null_pct_nonzero) * 0.4)))
    null_pct_nonzero.plot(kind="barh", color="#e74c3c", ax=ax)
    ax.set_xlabel("% de Valores Nulos")
    ax.set_title("Porcentaje de Valores Nulos por Columna")
    ax.axvline(x=50, color="#e67e22", linestyle="--", linewidth=1.5, label="Umbral 50%")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, "29_bar_nulos_porcentaje")
else:
    print("  ✅ No hay columnas con valores nulos en la muestra.")

📊 Gráfico 29: Porcentaje de nulos por columna
  ✅ No hay columnas con valores nulos en la muestra.


In [47]:
# === GRÁFICO 30: Box plots outliers ===
# Doble perspectiva: datos completos (para evidenciar outliers extremos)
# y recortados P1–P99 (para ver la distribución real).
print("📊 Gráfico 30: Box plots — detección de outliers extremos")

outlier_features = top_features[:6]
n_of = len(outlier_features)

fig, axes = plt.subplots(2, n_of, figsize=(4 * n_of, 12))

for i, feat in enumerate(outlier_features):
    data_out = df_sample_pd[feat].replace([np.inf, -np.inf], np.nan).dropna()

    ax_top = axes[0, i]
    ax_top.boxplot(data_out.values, vert=True, patch_artist=True,
                boxprops=dict(facecolor="#3498db", alpha=0.6),
                medianprops=dict(color="#e74c3c", linewidth=2))
    ax_top.set_title(f"{feat}\n(completo)", fontsize=9)
    ax_top.set_ylabel("Valor")

    q1, q99 = data_out.quantile(0.01), data_out.quantile(0.99)
    data_clip = data_out[(data_out >= q1) & (data_out <= q99)]
    ax_bot = axes[1, i]
    ax_bot.boxplot(data_clip.values, vert=True, patch_artist=True,
                boxprops=dict(facecolor="#2ecc71", alpha=0.6),
                medianprops=dict(color="#e74c3c", linewidth=2))
    ax_bot.set_title(f"{feat}\n(P1–P99)", fontsize=9)
    ax_bot.set_ylabel("Valor")

fig.suptitle("Box Plots — Detección de Outliers Extremos", fontsize=14, y=1.03)
fig.text(0.5, -0.01,
        "Fila superior: escala completa. Fila inferior: recorte P1–P99.",
        ha="center", fontsize=10, style="italic")
plt.tight_layout()
save_figure(fig, "30_boxplots_outliers")

📊 Gráfico 30: Box plots — detección de outliers extremos
  💾 Figura guardada: /home/hpc/22231088student/tfg_ids/figures/30_boxplots_outliers.png



---
## ✅ Resumen del EDA

| Bloque | Gráficos | Tipo | Ubicación |
|---|---|---|---|
| 2.1 Distribución de clases | 4 | Bar, log, donut (agrupado), treemap | Cuerpo |
| 2.2 Univariantes | ~8 | Hist+KDE (2 paneles), violin (facetado), box (2 filas), QQ (3 repr.), ridge (2 grupos) | Cuerpo + Anexo |
| 2.3 Bivariantes | 4+1 | Heatmap corr., hexbin, pair (estratificado), clustermap | Cuerpo + Scatter en Anexo |
| 2.4 Multivariante/DR | 5 | PCA 2D (elipses 95%), **t-SNE 3D**, scree, t-SNE 2D, parallel coord. | Cuerpo |
| 2.5 Temporal | 4 | Line, stacked area (abs + proporcional), heatmap | Cuerpo |
| 2.6 Perfiles de ataque | 3 | Radar, grouped bar (symlog), strip (winsorizado) | Cuerpo |
| 2.7 Calidad datos | 5 | Pipeline norm., stacked bar audit, msno matrix + tabla, bar nulos, box outliers (2 filas) | Cuerpo |
| **TOTAL** | **~33** | **Alta variedad + calidad profesional** | |

**Siguiente paso:** Notebook 3 — Limpieza y Preprocesamiento

In [48]:
spark.stop()
print("✅ Spark session cerrada. EDA completado.")

✅ Spark session cerrada. EDA completado.
